## Introduction

Our project aims to improve GPU task scheduling by building optimization models. It includes a basic model and an extended model, both formulated as MILP models. We aim to improve the overall system performance by efficiently assigning tasks to limited GPU resources. Under different settings, we aim to minimize the total completion time when all tasks must be executed, and maximize the total task value within a limited time when tasks can be skipped. 

In our basic model, we have 20 tasks and 4 identical GPUs. Each task has two possible configurations, each specifying the number of GPUs required and the runtime to complete the task. For example, a task can take 24 hours when using one GPU, but might only need 14 hours when running on two. Each task can choose only one configuration, and all tasks must be executed. We present the execution order by assigning a start time and a finish time to each task, and we ensure that the total number of GPUs in use does not exceed 4 at any time. Our goal is to choose the best configuration combination and the order of executing tasks to minimize the total completion time. 

In the extended model, each task is assigned a value to represent its priority. A higher value means a higher priority. Unlike the basic model, tasks are not required to be executed, and some tasks can be skipped as needed. In addition, there is a total execution time budget now. Our goal is to select a subset of tasks and their execution order to maximize the total value of completed tasks within the time limit. 

Mordern GPUs are widely used in large-scale computing. However, once a GPU task starts running, it usually cannot be interrupted. When multiple applications share the same GPU, important tasks may be delayed for a long time. Therefore, It is important to schedule tasks properly so that high-value tasks can be completed as early as possible. This can improve the overall system performance and make better use of expensive GPU resoueces. An early work, TimeGraph [1], propoeses a GPU scheduling method that uses task priorities to decide the execution order, with a max running time constraint to prevent one task from abusing resources. However, for simplicity, we do not consider the execution time limits in our project. Another recent work introduces the operation of a shared GPU cluster on HKUST campus [2]. GPUs are offered in packs of 1, 2, 4, and 8. While this strategy mitigates the fragmentation issues, it losses some scheduling flexibility and assumes a decent cluster.

We do not use real data for this project. Instead, we generate synthetic data for GPU requirements, runtime, and task values. We also consider the following assumptions to simplify our models:
 - All GPUs are identical and can run tasks at the same time.
 - Tasks are independent and do not have dependencies.
 - Tasks cannot be interrupted once it starts execution.
 - Each task can only choose one given configuration.
 - All tasks must be executed in the basic model.
 - Tasks can be skipped in the extended model. 


## Discussion

We have preliminary results with our basic model. We've formulated our basic model as follows:

#### 1. Sets and Parameters

First, we define the following sets:

* $I = \{1, ..., N\}$: the set of pending tasks
* $K = \{1, 2\}$: configuration options for each task
* $T = \{0, 1, ..., H - 1\}$: all discrete time periods in the planning space


Then, we define the parameters:

* $N$ = 20: number of pending tasks
* $G$ = 4: number of available GPUs
* $g_{ik} \in \{1, ..., G\}$: number of GPUs required by task $i$ under configuration $k \in \{1, 2\}$
* $p_{ik} \in \mathbb{Z_+}$: runtime of task $i$ under configuration $k$
* $H$: a sufficiently large time as the upperbound of the completion time: 
$$
H = \sum_{i \in I} \max_{k \in K} p_{ik}
$$

#### 2. Decision Variables

We define the final completion time as a variable:

$$
C_{max} \ge 0
$$

Then, use binary variables to define the start time and configuration decisions:

$$
x_{ikt} = \begin{cases}
1, \quad\text{if task $i$ starts at time $t$ with config $k$} \\
0, \quad\text{otherwise}
\end{cases}
$$

for all $i \in I$, $k \in K$, and $t = 0, ..., H - p_{ik}$.


#### 3. Objective and Constraints

We want to minimize the final completion time of the schedule:

$$
\begin{aligned}
\min \quad& C_{max} \\
\text{subject to} \quad
& \sum_{k \in K}\sum_{t=0}^{H-p_{ik}} x_{ikt} = 1 &\quad \forall i \in I &\quad \text{(task/config uniqueness)}\\
& \sum_{i \in I}\sum_{k \in K}\sum_{t \le \tau \le t + p_{ik}-1} g_{ik}x_{ikt} \le 4 &\quad \forall \tau \in T &\quad\text{(GPU capacity)}\\
& C_{max} \ge \sum_{k \in K}\sum_{t=0}^{H-p_{ik}} (t + p_{ik})x_{ikt} &\quad \forall i \in I &\quad\text{(final completion time)}\\
& x_{ikt} \in \{0, 1\} &\quad \forall i, k, t &\quad\text{(binary scheduling decisions)} \\
& C_{max} \ge 0 &&\quad\text{(non-negativity)} \\
\end{aligned}
$$




### Work Remaining

Since we have some results with the basic model, our primary focus will shift to formulating and solving our extended model. Additionally, we want to explore the tradeoff between the minimum completion time and the maximum task values. We plan to exam this idea by adjusting the objective function of our extended model. We expect to complete this part in one week.

We will also spend some efforts extending our results by conducting more experiments with more data. Particularly, we would like to explore:

1. How does a long-tail task with a long execution time, affect the effectiveness of the model?
2. Does this model solves the problem efficiently? We're interested in how much time does it take for an optimizer to solve the model. What will happen if we have a larger decision space?
3. What will happen if we feed *strange* configuration sets? For example, one task takes 4 hours using 1 GPU but requires 8 hours with 2 GPUs.

We expect this to be done in a few days before the report submission. We expect that we can complete the project within the two weeks before the project due date.

## Issues/concerns

## Reference

1. Kato, Shinpei, et al. "{TimeGraph}:{GPU} Scheduling for {Real-Time}{Multi-Tasking} Environments." 2011 USENIX Annual Technical Conference (USENIX ATC 11). 2011.
2. Xu, Kaiqiang, et al. "Design and operation of shared machine learning clusters on campus." Proceedings of the 30th ACM International Conference on Architectural Support for Programming Languages and Operating Systems, Volume 1. 2025.
